In [1]:
# ============================================================
# BUILD DIM_RACES — LOCAL GOLD
# ============================================================

import os
from pyspark.sql import functions as F

In [ ]:
try:
    SILVER_ROOT
except NameError:
    import nbformat
    from nbconvert import PythonExporter
    config_path = "/Users/manoharazalki/F1-Analytics/notebooks/00-common/01_environment_config.ipynb"
    with open(config_path, "r") as f:
        nb = nbformat.read(f, as_version=4)
    exporter = PythonExporter()
    source, _ = exporter.from_notebook_node(nb)
    exec(source)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/08 22:31:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/08 22:31:54 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


✔ Spark Session Initialized
✔ Environment Config Loaded
PROJECT_ROOT: /Users/manoharazalki/F1-ANALYTICS


In [3]:
silver_races = f"{SILVER_ROOT}/races/races_silver.parquet"
silver_circuits = f"{SILVER_ROOT}/circuits/circuits_silver.parquet"
gold_folder = f"{GOLD_ROOT}/dim_races"
gold_file = f"{gold_folder}/dim_races.parquet"
os.makedirs(gold_folder, exist_ok=True)

In [4]:
races_df = spark.read.parquet(silver_races)
circuits_df = spark.read.parquet(silver_circuits)

In [5]:
dim_races_df = (
    races_df.alias("r")
        .join(
            circuits_df.alias("c"),
            F.col("r.circuit_id") == F.col("c.circuit_id"),
            "left"
        )
        .select(
            F.col("r.year").alias("season"),
            F.col("r.round"),
            F.col("r.race_name"),
            F.col("r.race_date"),
            F.col("c.circuit_name"),
            F.col("c.locality"),
            F.col("c.country_name").alias("country_name")
        )
)

In [6]:
(
    dim_races_df
        .write
        .mode("overwrite")
        .parquet(gold_file)
)

print(f"✔ dim_races written to: {gold_file}")
spark.read.parquet(gold_file).show(10, truncate=False)

✔ dim_races written to: /Users/manoharazalki/F1-ANALYTICS/data/gold/dim_races/dim_races.parquet
+------+-----+------------------+----------+----------------------------+------------+------------+
|season|round|race_name         |race_date |circuit_name                |locality    |country_name|
+------+-----+------------------+----------+----------------------------+------------+------------+
|1950  |1    |British Grand Prix|1950-05-13|Silverstone Circuit         |Silverstone |Uk          |
|1950  |2    |Monaco Grand Prix |1950-05-21|Circuit De Monaco           |Monte Carlo |Monaco      |
|1950  |3    |Indianapolis 500  |1950-05-30|Indianapolis Motor Speedway |Indianapolis|Usa         |
|1950  |4    |Swiss Grand Prix  |1950-06-04|Circuit Bremgarten          |Bern        |Switzerland |
|1950  |5    |Belgian Grand Prix|1950-06-18|Circuit De Spa-francorchamps|Spa         |Belgium     |
|1950  |6    |French Grand Prix |1950-07-02|Reims-gueux                 |Reims       |France      |
|195